# Legoin – Tahap 3: Warna Kepingan dengan K-Means di CIELAB

**Kelompok 5 – Visual Komputer Cerdas** (Dika Aprilio Wibowo, Aldio Lisafron, Ricky Bonardo)

Notebook ini membaca warna tiap crop kepingan dari tahap 1 dan memetakannya ke salah satu **10 warna katalog** di kontrak API (`legoin-docs/api/contract.md`, bagian *The catalogue*), lengkap dengan **confidence 0–1**.

**Metode (yang paling sederhana dan masih masuk akal):**
1. **K-Means** (k = 4) mengelompokkan piksel crop di ruang warna **CIELAB**. Cluster yang lebih banyak berada di tengah crop daripada di tepinya dianggap kepingan, karena latar ada di tepi kotak detektor;
2. warna kepingan = median a\*, b\* dan persentil ke-75 L\* dari piksel kepingan. Persentil 75 dipakai karena sisi yang terkena cahaya lebih mewakili warna asli daripada sisi yang gelap;
3. bandingkan dengan **warna referensi** tiap id katalog memakai jarak **CIEDE2000 (ΔE)**, lalu ambil yang terdekat;
4. confidence = softmax dari −ΔE/T. Confidence dibatasi 0,5, sehingga kepingan ditandai untuk dicek, bila warna terdekat pun masih jauh (ΔE > batas) atau bila kepingan jelas berwarna tetapi tebakannya warna netral (mis. azure atau pink yang terdekat ke abu-abu terang). Ini untuk kepingan berwarna di luar katalog.

**Kenapa CIELAB, bukan HSV seperti di PRD?** K-Means tetap dipakai, tetapi jarak di CIELAB dibuat agar sebanding dengan beda warna yang dilihat mata. Hue di HSV tidak stabil untuk hitam, putih, dan abu-abu (saturasi mendekati 0), padahal 4 dari 10 warna katalog adalah warna netral.

**Kenapa referensi terdekat, bukan classifier?** Cara ini langsung jalan tanpa data berlabel (referensi awal dari Rebrickable), cukup dikalibrasi dengan sedikit label (10 titik warna), jaraknya memberi sinyal alami untuk "warna asing", dan backend cukup menyalin satu fungsi kecil. Classifier kecil (regresi logistik) tetap dibandingkan di bagian 7.

**Data:**
- Referensi awal: nilai RGB warna Rebrickable untuk 10 id warna katalog (https://rebrickable.com/colors/).
- Boiński, T. M., Zaraziński, S., & Śledź, B. (2021). *LEGO bricks for training classification network* (versi 1.1). MOST Wiedzy, Gdańsk University of Technology. https://doi.org/10.34808/rcza-jy08 (lisensi CC BY 4.0). Render di dataset ini menyimpan **nama warnanya di nama file** (43 warna, termasuk 10 warna katalog), sedangkan foto nyatanya **tidak punya label warna**.

**Runtime:** CPU sudah cukup; GPU tidak diperlukan.

**Alur:** Setup → Unduh subset data → Warna referensi → Metode → Evaluasi render → Foto nyata (tanpa label) → Crop foto HP berlabel → Simpan parameter

## 1. Setup lingkungan

In [ ]:
# Versi dikunci supaya hasil antar anggota kelompok sama (K-Means dan konversi warna)
%pip install -q scikit-image==0.26.0 scikit-learn==1.9.1 pandas matplotlib tqdm

In [ ]:
import sys
from pathlib import Path

# Google Colab: data di disk Colab (cepat), hasil di Google Drive (permanen).
# Lokal (VS Code/Jupyter): semua disimpan di folder notebook ini.
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content')
    SIMPAN_DIR = Path('/content/drive/MyDrive/VISKOM_Kelompok5')
else:
    ROOT = Path.cwd()
    SIMPAN_DIR = ROOT
SIMPAN_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = ROOT / 'data' / 'lego_klasifikasi'   # folder data yang sama dengan notebook tahap 2
RUNS_DIR = SIMPAN_DIR / 'runs'                  # crop tahap 1 ada di sini

## 2. Konfigurasi

Urutan dan nama `KATALOG_WARNA` adalah **id warna di kontrak API**. Nilai RGB-nya dari Rebrickable, sesuai id Rebrickable di kontrak.

In [ ]:
SEED = 42

# id warna kontrak -> (id Rebrickable, RGB Rebrickable)
KATALOG_WARNA = {
    'red': (4, 'C91A09'), 'blue': (1, '0055BF'), 'yellow': (14, 'F2CD37'), 'green': (2, '237841'),
    'black': (0, '05131D'), 'white': (15, 'FFFFFF'), 'light-gray': (71, 'A0A5A9'),
    'dark-gray': (72, '6C6E68'), 'orange': (25, 'FE8A18'), 'brown': (70, '582A12'),
}
LAIN = 'other'   # label untuk warna di luar katalog; tidak pernah dikirim ke web

# Nama warna LEGO di render MOST Wiedzy -> id katalog. 33 warna lainnya dianggap `other`.
# Light Bluish Gray = Medium Stone Grey, Dark Bluish Gray = Dark Stone Grey, Green (Rebrickable 2) = Dark Green.
PETA_WARNA_RENDER = {
    'Bright Red': 'red', 'Bright Blue': 'blue', 'Bright Yellow': 'yellow', 'Dark Green': 'green',
    'Black': 'black', 'White': 'white', 'Medium Stone Grey': 'light-gray', 'Dark Stone Grey': 'dark-gray',
    'Bright Orange': 'orange', 'Reddish Brown': 'brown',
}

# Folder 13 part katalog di dataset (sama dengan tahap 2); warnanya tidak bergantung pada jenis part
FOLDER_PART = ['3005', '3004', '3622', '3010', '3003', '3002', '3001',
               '3023', '3022', '3020', '3039', '3069b', '3062']
MAKS_RENDER_PER_FOLDER = 300   # ±90 render per warna katalog
MAKS_FOTO_PER_FOLDER = 12      # foto nyata hanya untuk pemeriksaan visual (tidak berlabel warna)

K_CLUSTER = 4             # latar, badan kepingan, sisi gelap, pantulan cahaya
UKURAN_ANALISIS = 96      # crop diperkecil dulu: cukup untuk warna dan jauh lebih cepat
RASIO_KALIBRASI = 0.5     # porsi render untuk kalibrasi; sisanya untuk uji
MIN_PER_WARNA = 5         # minimal contoh berlabel agar referensi suatu warna dikalibrasi
KROMA_NETRAL = 15         # kroma C* di bawah ini = warna netral (hitam, putih, abu-abu)
BATAS_YAKIN = 0.6         # LOW_CONFIDENCE di web app

## 3. Unduh subset dataset

Zip dataset berukuran ±6 GB, padahal kita hanya butuh 13 dari 447 folder. Server MOST Wiedzy (S3) mendukung **HTTP range request**, jadi:
1. `zipfile` membaca daftar isi zip (central directory, beberapa MB di akhir file) lewat objek `BerkasHTTP` yang bisa di-*seek*;
2. isi tiap folder tersimpan berurutan di dalam zip, sehingga satu folder cukup diunduh dengan **satu request** rentang byte;
3. tiap gambar di-resize ke sisi terpanjang 256 px lalu disimpan sebagai JPEG.

Gambar yang sudah ada dilewati, jadi cell ini aman dijalankan ulang. Folder datanya sama dengan notebook tahap 2, jadi gambar yang sudah diunduh di sana tidak diunduh ulang.

In [ ]:
import io
import struct
import time
import urllib.request
import zipfile
import zlib
from bisect import bisect_right
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

from PIL import Image, ImageOps
from tqdm.auto import tqdm

URL_DATASET = ('https://mostwiedzy.pl/en/open-research-data/'
               'lego-bricks-for-training-classification-network,202309140842198941751-0/download')
UKURAN_SIMPAN = 256   # sisi terpanjang gambar yang disimpan (cukup untuk input 224)


def minta_rentang(url, awal, akhir, coba=5):
    # GET dengan header Range, diulang bila koneksi terputus.
    for i in range(coba):
        try:
            req = urllib.request.Request(url, headers={'Range': f'bytes={awal}-{akhir}'})
            with urllib.request.urlopen(req, timeout=120) as r:
                return r.geturl(), r.headers.get('Content-Range'), r.read()
        except OSError:
            if i == coba - 1:
                raise
            time.sleep(2 ** i)


class BerkasHTTP:
    # Berkas jarak jauh yang bisa di-seek, cukup untuk zipfile membaca daftar isi zip.

    def __init__(self, url):
        self.url, rentang, _ = minta_rentang(url, 0, 0)   # url S3 setelah redirect (berlaku 24 jam)
        self.size = int(rentang.split('/')[-1])
        self.pos = 0

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        self.pos = (offset, self.pos + offset, self.size + offset)[whence]
        return self.pos

    def read(self, n=-1):
        n = self.size - self.pos if n is None or n < 0 else min(n, self.size - self.pos)
        if n <= 0:
            return b''
        data = minta_rentang(self.url, self.pos, self.pos + n - 1)[2]
        self.pos += len(data)
        return data


def isi_anggota(potongan, awal, info):
    # Ambil isi satu file dari potongan byte zip yang dimulai di offset `awal`.
    p = info.header_offset - awal
    n_nama, n_extra = struct.unpack('<2H', potongan[p + 26:p + 30])
    mulai = p + 30 + n_nama + n_extra
    data = potongan[mulai:mulai + info.compress_size]
    return zlib.decompress(data, -15) if info.compress_type == zipfile.ZIP_DEFLATED else data


def path_lokal(info):
    sumber, folder, nama = info.filename.split('/')
    return DATA_DIR / sumber / folder / (Path(nama).stem + '.jpg')


def simpan_gambar(data, tujuan):
    im = ImageOps.exif_transpose(Image.open(io.BytesIO(data))).convert('RGB')
    im.thumbnail((UKURAN_SIMPAN, UKURAN_SIMPAN), Image.LANCZOS)
    tujuan.parent.mkdir(parents=True, exist_ok=True)
    im.save(tujuan, quality=92)


print('Membaca daftar isi zip ...')
berkas_zip = BerkasHTTP(URL_DATASET)
daftar_zip = zipfile.ZipFile(berkas_zip)
ISI_FOLDER = defaultdict(list)   # (photos|renders, folder) -> [ZipInfo] dalam urutan zip
for info in daftar_zip.infolist():
    bagian = info.filename.split('/')
    if len(bagian) == 3 and bagian[2]:
        ISI_FOLDER[(bagian[0], bagian[1])].append(info)
OFFSET_URUT = sorted(i.header_offset for infos in ISI_FOLDER.values() for i in infos) + [daftar_zip.start_dir]
print(f'{berkas_zip.size / 1e9:.1f} GB, {len(OFFSET_URUT) - 1} gambar,',
      len({f for _, f in ISI_FOLDER}), 'folder part')


def unduh_folder(anggota):
    # Unduh file-file `anggota` (satu folder) dengan satu request rentang byte.
    kurang = [i for i in anggota if not path_lokal(i).exists()]
    if not kurang:
        return 0
    kurang.sort(key=lambda i: i.header_offset)
    awal = kurang[0].header_offset
    akhir = OFFSET_URUT[bisect_right(OFFSET_URUT, kurang[-1].header_offset)]
    potongan = minta_rentang(berkas_zip.url, awal, akhir - 1)[2]
    for info in kurang:
        simpan_gambar(isi_anggota(potongan, awal, info), path_lokal(info))
    return len(kurang)


def unduh_semua(pilihan, desc):
    with ThreadPoolExecutor(max_workers=4) as ex:
        n = sum(tqdm(ex.map(unduh_folder, pilihan), total=len(pilihan), desc=desc))
    print(desc, ':', n, 'gambar baru diunduh')

In [ ]:
import pandas as pd

pilihan = {}
for folder in FOLDER_PART:
    pilihan[('renders', folder)] = ISI_FOLDER[('renders', folder)][:MAKS_RENDER_PER_FOLDER]
    pilihan[('photos', folder)] = ISI_FOLDER[('photos', folder)][:MAKS_FOTO_PER_FOLDER]
unduh_semua(list(pilihan.values()), 'unduh')

rows = []
for (sumber, folder), infos in pilihan.items():
    for info in infos:
        nama = Path(info.filename).stem
        # Render: <design id>_<nama warna>_<urutan>_<timestamp>
        warna_asli = nama.split('_')[1] if sumber == 'renders' else None
        rows.append({'path': path_lokal(info), 'sumber': sumber, 'folder': folder, 'warna_asli': warna_asli,
                     'y': PETA_WARNA_RENDER.get(warna_asli, LAIN) if warna_asli else None})
df = pd.DataFrame(rows)
render = df[df.sumber == 'renders'].reset_index(drop=True)
foto = df[df.sumber == 'photos'].reset_index(drop=True)
print(len(render), 'render dan', len(foto), 'foto nyata')
render.y.value_counts()

## 4. Warna referensi

Titik awal adalah warna resmi Rebrickable. Kamera, cahaya, dan bayangan menggeser warna yang terlihat, jadi referensi ini nanti **dikalibrasi** dengan crop berlabel: referensi tiap warna diganti dengan rata-rata warna kepingan berlabel warna itu.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import skimage
import sklearn
from IPython.display import display
from skimage.color import deltaE_ciede2000, lab2rgb, rgb2lab
from sklearn.cluster import KMeans

ID_WARNA = list(KATALOG_WARNA)


def hex_ke_lab(h):
    rgb = np.array([[[int(h[i:i + 2], 16) / 255 for i in (0, 2, 4)]]])
    return rgb2lab(rgb)[0, 0]


def tampilkan_referensi(daftar_ref, judul):
    # Kotak warna tiap referensi, satu baris per set referensi.
    _, axes = plt.subplots(len(daftar_ref), 1, figsize=(12, 1.3 * len(daftar_ref)), squeeze=False)
    for ax, ref, j in zip(axes[:, 0], daftar_ref, judul):
        ax.imshow(np.clip(lab2rgb(ref[None]), 0, 1), aspect='auto')
        ax.set_xticks(range(len(ID_WARNA)), ID_WARNA)
        ax.set_yticks([])
        ax.set_ylabel(j, rotation=0, ha='right', va='center')
    plt.tight_layout()
    plt.show()


REF_REBRICKABLE = np.array([hex_ke_lab(rgb) for _, rgb in KATALOG_WARNA.values()])
tampilkan_referensi([REF_REBRICKABLE], ['Rebrickable'])
pd.DataFrame(REF_REBRICKABLE, index=ID_WARNA, columns=['L*', 'a*', 'b*']).round(1)

## 5. Metode

Fungsi di cell berikut adalah **implementasi acuan untuk backend**: crop dari kotak detektor masuk, id warna dan confidence keluar. Parameternya (referensi, T, batas ΔE) disimpan di bagian 9.

In [ ]:
def warna_kepingan(im, k=K_CLUSTER, ukuran=UKURAN_ANALISIS):
    # Warna kepingan (L*, a*, b*) dari satu crop, plus mask piksel yang dianggap kepingan.
    im = im.convert('RGB')
    im.thumbnail((ukuran, ukuran))
    lab = rgb2lab(np.asarray(im) / 255.0)
    h, w = lab.shape[:2]
    px = lab.reshape(-1, 3)
    # Bobot tengah (Gaussian) dan cincin tepi 8%: kotak detektor berpusat di kepingan, latar ada di tepi.
    yy, xx = np.mgrid[:h, :w]
    tengah = np.exp(-(((xx - (w - 1) / 2) / (0.3 * w)) ** 2 + ((yy - (h - 1) / 2) / (0.3 * h)) ** 2) / 2).ravel()
    t = max(1, round(min(h, w) * 0.08))
    tepi = np.ones((h, w), bool)
    tepi[t:h - t, t:w - t] = False
    tepi = tepi.ravel()
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')   # crop dengan kurang dari k warna berbeda
        label = KMeans(k, n_init=2, random_state=0).fit_predict(px)
    # Skor cluster = porsi bobot tengah - porsi cincin tepi; skor > 0 berarti bagian kepingan.
    skor = np.array([tengah[label == c].sum() / tengah.sum() - (label[tepi] == c).mean() for c in range(k)])
    kepingan = np.flatnonzero(skor > 0) if (skor > 0).any() else [int(skor.argmax())]
    mask = np.isin(label, kepingan)
    f = px[mask]
    warna = np.array([np.percentile(f[:, 0], 75), np.median(f[:, 1]), np.median(f[:, 2])])
    return warna, mask.reshape(h, w)


def jarak(lab, ref):
    # CIEDE2000 dari tiap warna (n, 3) ke tiap referensi (m, 3) -> (n, m).
    lab = np.atleast_2d(lab)
    return np.stack([deltaE_ciede2000(lab, np.broadcast_to(r, lab.shape)) for r in ref], 1)


def putuskan(lab, ref, suhu, d_maks):
    # Dari warna kepingan (n, 3) dan referensi (10, 3): id warna katalog dan confidence 0..1.
    lab = np.atleast_2d(lab)
    d = jarak(lab, ref)
    z = -d / suhu
    p = np.exp(z - z.max(1, keepdims=True))
    p /= p.sum(1, keepdims=True)
    i = p.argmax(1)
    # Warna asing: terlalu jauh dari semua referensi, atau jelas berwarna tetapi tebakannya warna netral.
    kroma = np.hypot(lab[:, 1], lab[:, 2])
    netral = np.hypot(ref[i, 1], ref[i, 2]) < KROMA_NETRAL
    asing = (d.min(1) > d_maks) | ((kroma >= KROMA_NETRAL) & netral)
    return np.array(ID_WARNA)[i], np.where(asing, np.minimum(p.max(1), 0.5), p.max(1))


def kalibrasi(lab, y):
    # Referensi = rata-rata warna kepingan berlabel; warna yang contohnya kurang tetap memakai Rebrickable.
    ref = REF_REBRICKABLE.copy()
    for j, c in enumerate(ID_WARNA):
        if (y == c).sum() >= MIN_PER_WARNA:
            ref[j] = lab[y == c].mean(0)
    return ref


def cari_suhu(d, y):
    # T yang meminimalkan negative log-likelihood warna benar, dan batas ΔE (persentil 95 jarak ke warna benar).
    m = np.isin(y, ID_WARNA)
    j = np.array([ID_WARNA.index(c) for c in y[m]])
    dm = d[m]
    benar = dm[np.arange(len(j)), j]
    kandidat = np.geomspace(0.5, 50, 60)
    nll = [np.mean(benar / t + np.log(np.exp(-dm / t).sum(1))) for t in kandidat]
    return float(kandidat[int(np.argmin(nll))]), float(np.percentile(benar, 95))


def wilson(k, n, z=1.96):
    # Selang kepercayaan 95% untuk proporsi k/n.
    if n == 0:
        return (float('nan'), float('nan'))
    p, dd = k / n, 1 + z * z / n
    c = (p + z * z / (2 * n)) / dd
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / dd
    return (c - h, c + h)


def nilai(y, pred, conf):
    # Akurasi pada warna katalog (+ CI 95%) dan seberapa sering warna di luar katalog ditandai.
    m = np.isin(y, ID_WARNA)
    k, n = int((pred[m] == y[m]).sum()), int(m.sum())
    lo, hi = wilson(k, n)
    yakin = m & (conf >= BATAS_YAKIN)
    return {'n warna katalog': n, 'akurasi': k / n if n else float('nan'), 'CI 95% bawah': lo, 'CI 95% atas': hi,
            f'porsi conf >= {BATAS_YAKIN}': yakin.sum() / n if n else float('nan'),
            f'akurasi saat conf >= {BATAS_YAKIN}': (pred[yakin] == y[yakin]).mean() if yakin.any() else float('nan'),
            'n warna luar katalog': int((~m).sum()),
            'warna luar katalog ditandai': (conf[~m] < BATAS_YAKIN).mean() if (~m).any() else float('nan')}

In [ ]:
contoh = render.groupby('y').head(1).head(6)
fig, axes = plt.subplots(3, len(contoh), figsize=(3 * len(contoh), 8))
for kol, r in enumerate(contoh.itertuples()):
    im = Image.open(r.path).convert('RGB')
    lab, mask = warna_kepingan(im)
    pred, conf = putuskan(lab, REF_REBRICKABLE, 5.0, 30.0)
    axes[0, kol].imshow(im)
    axes[0, kol].set_title(f'{r.warna_asli}\n(label: {r.y})', fontsize=9)
    axes[1, kol].imshow(mask, cmap='gray')
    axes[1, kol].set_title('piksel kepingan', fontsize=9)
    axes[2, kol].imshow(np.clip(lab2rgb(lab[None, None]), 0, 1)[0, 0][None, None].repeat(20, 0).repeat(20, 1))
    axes[2, kol].set_title(f'-> {pred[0]}', fontsize=9)
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Ekstraksi warna semua crop

In [ ]:
def warna_banyak(paths):
    return np.array([warna_kepingan(Image.open(p))[0] for p in tqdm(paths, desc='warna')])


render_lab = warna_banyak(render.path)
foto_lab = warna_banyak(foto.path)

## 7. Evaluasi pada render (berlabel, tetapi sintetis)

Render dibagi acak menjadi data **kalibrasi** dan **uji**. Tiga cara dibandingkan:
- **A. Referensi Rebrickable** apa adanya (tanpa data berlabel; T dan batas ΔE tetap dicari pada data kalibrasi);
- **B. Referensi terkalibrasi**: rata-rata warna render berlabel tiap warna;
- **C. Classifier kecil**: regresi logistik pada (L\*, a\*, b\*) warna kepingan, dilatih pada render warna katalog yang sama dengan B.

**Batasan penting.** Warna render dibuat dari kode hex (mis. *Bright Red* = `FF0000`) di bawah cahaya render, bukan plastik LEGO asli di bawah lampu rumah. Angka di sini membuktikan **pipeline berjalan** dan menguji penandaan warna asing (33 warna render di luar katalog), tetapi **bukan** akurasi warna pada foto nyata yang ditargetkan PRD (≥ 85%).

In [ ]:
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(SEED)
acak = rng.permutation(len(render))
n_kal = round(len(render) * RASIO_KALIBRASI)
kal, uji = acak[:n_kal], acak[n_kal:]
y_r = render.y.to_numpy()

hasil = {}
suhu_a, dmaks_a = cari_suhu(jarak(render_lab[kal], REF_REBRICKABLE), y_r[kal])
pred_a, conf_a = putuskan(render_lab[uji], REF_REBRICKABLE, suhu_a, dmaks_a)
hasil['A. Rebrickable'] = nilai(y_r[uji], pred_a, conf_a)

ref_render = kalibrasi(render_lab[kal], y_r[kal])
suhu_b, dmaks_b = cari_suhu(jarak(render_lab[kal], ref_render), y_r[kal])
pred_b, conf_b = putuskan(render_lab[uji], ref_render, suhu_b, dmaks_b)
hasil['B. terkalibrasi render'] = nilai(y_r[uji], pred_b, conf_b)

def regresi_logistik(lab_latih, y_latih, lab_uji):
    # Classifier kecil, dilatih hanya pada warna katalog (data yang sama dengan kalibrasi B).
    m = np.isin(y_latih, ID_WARNA)
    if len(set(y_latih[m])) < 2:
        return np.full(len(lab_uji), None), np.zeros(len(lab_uji))
    lr = LogisticRegression(max_iter=5000).fit(lab_latih[m], y_latih[m])
    proba = lr.predict_proba(lab_uji)
    return lr.classes_[proba.argmax(1)], proba.max(1)


pred_c, conf_c = regresi_logistik(render_lab[kal], y_r[kal], render_lab[uji])
hasil['C. regresi logistik'] = nilai(y_r[uji], pred_c, conf_c)

print(f'A: T = {suhu_a:.1f}, batas ΔE = {dmaks_a:.1f} | B: T = {suhu_b:.1f}, batas ΔE = {dmaks_b:.1f}')
pd.DataFrame(hasil).round(3)

In [ ]:
tampilkan_referensi([REF_REBRICKABLE, ref_render], ['Rebrickable', 'render'])

m = np.isin(y_r[uji], ID_WARNA)
cm = pd.crosstab(pd.Series(y_r[uji][m], name='sebenarnya'), pd.Series(pred_a[m], name='prediksi A'))
cm.reindex(index=ID_WARNA, columns=ID_WARNA, fill_value=0)

Baris referensi "render" memperlihatkan seberapa jauh cahaya render menggeser warna dari nilai resmi, terutama kecerahan abu-abu dan putih. Hal yang sama akan terjadi pada foto HP dengan cahaya yang berbeda-beda, dan itulah alasan referensi **harus dikalibrasi pada crop foto nyata**, bukan pada render. Referensi "render" tidak diekspor.

Classifier C biasanya lebih akurat dari B pada render, karena ia bisa mempelajari batas yang tidak bulat di antara warna. Tetapi ia dilatih pada cahaya render, dan tanpa kelas `other` ia sulit menandai warna asing. Keunggulannya baru berarti bila terulang pada crop foto nyata berlabel (bagian 8.2); sampai saat itu yang diekspor adalah metode referensi.

## 8. Foto nyata

### 8.1 Foto studio MOST Wiedzy (tanpa label warna)

Foto nyata di dataset tidak punya label warna, jadi bagian ini hanya **pemeriksaan visual**: judul merah berarti confidence < 0,6 (ditandai untuk dicek). Banyak kepingan di foto ini berwarna di luar katalog (tan, azure, lime, merah tua, transparan, krom), jadi tanda merah di sana memang diharapkan.

In [ ]:
pred_f, conf_f = putuskan(foto_lab, REF_REBRICKABLE, suhu_a, dmaks_a)
tampil = foto.sample(min(24, len(foto)), random_state=SEED).index
fig, axes = plt.subplots(3, 8, figsize=(20, 8.5))
for ax in axes.flat:
    ax.axis('off')
for ax, i in zip(axes.flat, tampil):
    ax.imshow(Image.open(foto.path[i]))
    ax.set_title(f'{pred_f[i]} ({conf_f[i]:.2f})', fontsize=10, color='black' if conf_f[i] >= BATAS_YAKIN else 'red')
fig.suptitle('Foto nyata tanpa label: prediksi dengan referensi Rebrickable')
plt.tight_layout()
plt.show()
print(f'Porsi foto yang ditandai: {(conf_f < BATAS_YAKIN).mean():.1%}')

### 8.2 Crop foto HP berlabel (target PRD)

Belum ada data publik berisi foto nyata yang dilabeli dengan 10 warna katalog: foto MOST Wiedzy tidak berlabel warna, dataset tahap 1 hanya berlabel `lego`, dan B200C berupa render. Jadi **akurasi warna pada foto nyata belum bisa diukur**. Tim perlu melabeli sendiri satu set kecil.

**Rencana pelabelan** (file yang sama dengan tahap 2: `data/foto_mandiri/label.csv`, kolom `file,part,color`):
1. Foto kepingan milik kelompok dengan HP, di berbagai latar dan cahaya (siang dekat jendela, lampu putih, lampu kuning). Target minimal **30 crop per warna katalog** (±300) dan ±50 kepingan berwarna di luar katalog.
2. Potong dengan detektor tahap 1 (`save_crop=True`), lalu isi kolom `color` dengan id warna kontrak, atau `other` untuk warna di luar katalog.
3. Labeli sambil melihat kepingan aslinya, jangan hanya fotonya, dengan swatch Rebrickable di bagian 4 sebagai acuan. Pasangan yang paling sering tertukar adalah `light-gray`/`white`, `dark-gray`/`black`, `brown`/`dark-gray`, dan `orange`/`yellow`.
4. Dua anggota melabeli secara terpisah; bila berbeda, putuskan bersama atau buang crop itu.
5. Jalankan cell berikut. Akurasi diukur dengan **validasi silang 5 lipatan**: referensi dikalibrasi pada 4/5 data dan diuji pada 1/5 sisanya, bergantian. Referensi akhir yang diekspor dikalibrasi pada semua label.

In [ ]:
LABEL_CSV = ROOT / 'data' / 'foto_mandiri' / 'label.csv'

REF_EKSPOR, SUMBER_REF, SUHU, D_MAKS = REF_REBRICKABLE, 'rebrickable', suhu_a, dmaks_a
hasil_hp = None
if LABEL_CSV.exists():
    lab_csv = pd.read_csv(LABEL_CSV).dropna(subset=['color'])
    hp_lab = warna_banyak([LABEL_CSV.parent / f for f in lab_csv.file])
    y_hp = lab_csv.color.to_numpy()
    lipatan = np.random.default_rng(SEED).permutation(len(y_hp)) % 5
    hasil_cv = {m: (np.empty(len(y_hp), object), np.zeros(len(y_hp))) for m in 'ABC'}
    for f in range(5):
        tr, te = lipatan != f, lipatan == f
        s, dm = cari_suhu(jarak(hp_lab[tr], REF_REBRICKABLE), y_hp[tr])
        hasil_cv['A'][0][te], hasil_cv['A'][1][te] = putuskan(hp_lab[te], REF_REBRICKABLE, s, dm)
        ref = kalibrasi(hp_lab[tr], y_hp[tr])
        s, dm = cari_suhu(jarak(hp_lab[tr], ref), y_hp[tr])
        hasil_cv['B'][0][te], hasil_cv['B'][1][te] = putuskan(hp_lab[te], ref, s, dm)
        hasil_cv['C'][0][te], hasil_cv['C'][1][te] = regresi_logistik(hp_lab[tr], y_hp[tr], hp_lab[te])
    hasil_hp = pd.DataFrame({
        'A. Rebrickable': nilai(y_hp, *hasil_cv['A']),
        'B. terkalibrasi foto': nilai(y_hp, *hasil_cv['B']),
        'C. regresi logistik': nilai(y_hp, *hasil_cv['C']),
    }).round(3)
    REF_EKSPOR, SUMBER_REF = kalibrasi(hp_lab, y_hp), LABEL_CSV.name
    SUHU, D_MAKS = cari_suhu(jarak(hp_lab, REF_EKSPOR), y_hp)
    tampilkan_referensi([REF_REBRICKABLE, REF_EKSPOR], ['Rebrickable', 'foto HP'])
    display(hasil_hp)
else:
    print('Belum ada', LABEL_CSV, '-> akurasi warna pada foto nyata belum bisa diukur.')
    print('Parameter yang diekspor memakai referensi Rebrickable; T dan batas ΔE dari render (sementara).')

## 9. Simpan parameter untuk backend

Tahap ini tidak punya bobot jaringan. Backend cukup menyalin fungsi `warna_kepingan`, `jarak`, dan `putuskan` dari bagian 5, lalu memuat parameter dari `legoin_warna.json`. Confidence akhir sebuah deteksi di kontrak adalah satu angka untuk part dan warna sekaligus; nilai terkecil dari confidence tahap 2 dan tahap 3 adalah pilihan yang aman.

In [ ]:
import json

PARAM = SIMPAN_DIR / 'legoin_warna.json'
param = {
    'method': 'k-means in CIELAB on the crop; brick = clusters with centre weight > border share; '
              'colour = 75th percentile L*, median a*, median b* of brick pixels; nearest reference by CIEDE2000',
    'colors': ID_WARNA,                                   # id warna kontrak API
    'rebrickable_ids': {c: rb for c, (rb, _) in KATALOG_WARNA.items()},
    'reference_lab': {c: [round(float(v), 3) for v in REF_EKSPOR[j]] for j, c in enumerate(ID_WARNA)},
    'reference_source': SUMBER_REF,
    'temperature': round(SUHU, 4),
    'max_delta_e': round(D_MAKS, 3),
    'confidence': 'softmax(-deltaE / temperature) over the 10 colours; at most 0.5 when the smallest '
                  'deltaE > max_delta_e, or when the brick chroma >= neutral_chroma but the chosen '
                  'reference chroma < neutral_chroma',
    'neutral_chroma': KROMA_NETRAL,
    'low_confidence': BATAS_YAKIN,
    'kmeans_k': K_CLUSTER,
    'kmeans_n_init': 2,
    'kmeans_random_state': 0,
    'analysis_size': UKURAN_ANALISIS,
    'lightness_percentile': 75,
    'border_fraction': 0.08,
    'centre_sigma': 0.3,
    'scikit_learn': sklearn.__version__,
    'scikit_image': skimage.__version__,
}
PARAM.write_text(json.dumps(param, indent=2))
print('Parameter disimpan ke', PARAM)

# Cek: baca ulang seperti backend
p = json.loads(PARAM.read_text())
ref = np.array([p['reference_lab'][c] for c in p['colors']])
lab, _ = warna_kepingan(Image.open(render.path[0]))
print('Contoh:', render.warna_asli[0], '->', putuskan(lab, ref, p['temperature'], p['max_delta_e']))

## 10. Catatan dan langkah berikutnya

- Catat ke laporan: tabel bagian 7 (render) dengan keterangan **sintetis**, dan tabel bagian 8.2 bila `label.csv` sudah ada. Target PRD (≥ 85%) hanya boleh diklaim dari bagian 8.2.
- Kepingan berwarna di luar katalog itu umum (lihat bagian 8.1). Tim perlu sepakat apakah kepingan itu dihitung di metrik 85% atau dilaporkan terpisah sebagai "ditandai dengan benar"; notebook ini melaporkannya terpisah.
- Bila cahaya foto HP sangat bervariasi, coba *white balance* sederhana sebelum K-Means, misalnya dengan selembar kertas putih di foto sebagai acuan.
- Eksperimen yang layak dibandingkan: `K_CLUSTER` 3 vs 4, ΔE2000 vs jarak Euclid di CIELAB (ΔE76), dan regresi logistik (C) pada crop foto nyata.